# Reproduction of paper v9

*Human Medians and a Language-Model Forecast: Encompassing and Combination on One ForecastBench Round*
(v9, https://doi.org/10.5281/zenodo.23147918).

This notebook reruns the locked pipeline from the raw data and checks every number the paper
reports against the recomputed value.

1. **Provenance.** The datasets repository must be at `config.DATASETS_COMMIT` and both tarballs must
   hash to `config.TARBALL_SHA256`. The notebook stops otherwise.
2. **Pipeline.** `full.py` (SPEC v1), `v2_full.py` (SPEC v2), `amendment4_h5.py`, `amendment5.py`,
   `amendment5b.py` and `amendment6.py` run in that order, unedited. All but `amendment4_h5` are scripts
   with top-level code, so each runs in a subprocess and the notebook reads its report.
   `amendment4_h5` has a `main()` and is imported. The only intervention is that `config.DOCS`,
   `config.FIGDIR` and `config.DERIVED` are pointed at a fresh run directory outside the repository
   (`config.STUDY_ROOT / "repro_v9"`). That keeps the committed reports untouched and prevents
   `v2_full.py` from reusing checkpoints from an earlier run.
3. **Checks the modules do not print.** Some numbers were computed after Amendment 6 as read-only checks
   (`docs/v9_checks_NOTE.md`): the reverse encompassing reading, the superforecaster-id counts and the
   leave-one-id-out footnote. The notebook recomputes them here from the pipeline's own functions
   (`fbdata`, `models`, `v2common`, and the point-fit part of `amendment6.py`).
4. **Number-by-number check.** Every paper number is transcribed in `notebooks/paper_numbers_v9.json`
   with its section, and compared with the recomputed value. A number matches when the recomputed value
   rounds to it, i.e. when it lies within half a unit of the last printed digit.

Outputs are aggregates only. No raw forecast rows are printed.

In [1]:
import ast
import hashlib
import importlib
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
import matplotlib


def repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "code" / "config.py").exists():
            return p
    raise RuntimeError("Run this notebook from inside the forecast-complementarity repository.")


REPO = repo_root()
CODE = REPO / "code"
sys.path.insert(0, str(CODE))
import config as CFG

# Seeds. Every module fixes its own seeds from config / the SPEC; the notebook's own computations are
# deterministic, but the global generators and the subprocess hash seed are fixed here as well.
SEED = CFG.SPLIT_HALF_SEED
random.seed(SEED)
np.random.seed(SEED)
ENV = dict(os.environ, PYTHONHASHSEED=str(SEED), MPLBACKEND="Agg")
# The modules honour these for smoke tests only; the SPEC values are the defaults, so clear them.
for k in ("V2_TAG", "V2_DRAWS", "V2_MAXVAR", "A6_DRAWS", "A6_NOISE", "A2_DRAWS", "A2_TAG"):
    ENV.pop(k, None)
    os.environ.pop(k, None)

T_START = time.time()
print("repo        ", REPO)
print("study root  ", CFG.STUDY_ROOT)
print("repo HEAD   ", subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True,
                                     text=True).stdout.strip())
print("python      ", platform.python_version(), "|", platform.platform())
for m in (np, pd, scipy, statsmodels, matplotlib):
    print(f"{m.__name__:12s}", m.__version__)
print("seed        ", SEED, "(config.SPLIT_HALF_SEED; PYTHONHASHSEED for subprocesses)")

repo         /Users/aaron/Desktop/forecast-complementarity
study root   /Users/aaron/Desktop/forecast-study
repo HEAD    2283f7e125804a174ca9f73ba4e643f46c600191
python       3.9.6 | macOS-27.0.1-arm64-arm-64bit
numpy        2.0.2
pandas       2.3.3
scipy        1.13.1
statsmodels  0.14.6
matplotlib   3.9.4
seed         20260906 (config.SPLIT_HALF_SEED; PYTHONHASHSEED for subprocesses)


## 1. Provenance

The pipeline reads the human forecasts and resolutions from the `forecastbench-datasets` checkout,
and the model forecasts from the extracted `processed_forecast_sets.tar.gz`. The checks below are:

- the checkout is at `config.DATASETS_COMMIT`, and its working tree differs from that commit only in the
  Git LFS file recorded in `data/PROVENANCE.md` §2, which must hash to the recorded value;
- both tarballs hash to `config.TARBALL_SHA256`;
- every extracted model file for the round is byte-identical to its member in the hashed tarball.

In [2]:
class ProvenanceError(RuntimeError):
    pass


def stop(msg):
    raise ProvenanceError(msg + "\nFetch the data as described in data/PROVENANCE.md "
                          "(README, 'Reproduce') and rerun.")


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


RAW = CFG.STUDY_ROOT / "data" / "raw"
TARBALLS = {name: RAW / name for name in CFG.TARBALL_SHA256}
# data/PROVENANCE.md §2, "Git LFS caveat": the public individual file fetched outside git.
LFS_FILE = CFG.HUMAN_FILES["P"]
LFS_SHA256 = "4b3661081a54be787832f89051b19c2ada2f8c96396d692db3f5a753bec2b1fd"

required = [CFG.STUDY_ROOT, CFG.DATASETS_REPO, CFG.RESOLUTION_SET, *CFG.HUMAN_FILES.values(),
            CFG.MODEL_DIR, *TARBALLS.values()]
missing = [str(p) for p in required if not p.exists()]
if missing:
    stop("Data absent. Missing:\n  " + "\n  ".join(missing))

# (a) datasets commit
git = lambda *a: subprocess.run(["git", "-C", str(CFG.DATASETS_REPO), *a],
                                capture_output=True, text=True)
head = git("rev-parse", "HEAD").stdout.strip()
if head != CFG.DATASETS_COMMIT:
    stop(f"forecastbench-datasets is at {head or '(not a git checkout)'}, "
         f"expected config.DATASETS_COMMIT = {CFG.DATASETS_COMMIT}.")
top = Path(git("rev-parse", "--show-toplevel").stdout.strip())
dirty = [l[3:] for l in git("status", "--porcelain", "--untracked-files=no").stdout.splitlines()]
allowed = {str(LFS_FILE.relative_to(top))}
if set(dirty) - allowed:
    stop("forecastbench-datasets has local modifications beyond the recorded LFS file:\n  "
         + "\n  ".join(sorted(set(dirty) - allowed)))
if LFS_FILE.read_bytes()[:40].startswith(b"version https://git-lfs"):
    stop(f"{LFS_FILE.name} is a Git LFS pointer, not the data (see PROVENANCE.md §2).")
if sha256(LFS_FILE) != LFS_SHA256:
    stop(f"{LFS_FILE.name} does not hash to the value recorded in PROVENANCE.md §2.")
print(f"datasets commit  {head}  == config.DATASETS_COMMIT  OK")
print(f"LFS file         {LFS_FILE.name}  sha256 matches PROVENANCE.md §2  OK")

# (b) tarball hashes
for name, path in TARBALLS.items():
    got = sha256(path)
    if got != CFG.TARBALL_SHA256[name]:
        stop(f"{name}: sha256 {got} != config.TARBALL_SHA256 {CFG.TARBALL_SHA256[name]}.")
    print(f"{name:32s} sha256 {got}  OK")

# (c) the extracted model files the pipeline reads are the tarball's bytes
prefix = f"{CFG.PROCESSED.name}/{CFG.ROUND}/"
in_tar = {}
with tarfile.open(TARBALLS["processed_forecast_sets.tar.gz"], "r|gz") as tf:
    for mem in tf:
        if mem.isfile() and mem.name.startswith(prefix):
            in_tar[mem.name[len(prefix):]] = hashlib.sha256(tf.extractfile(mem).read()).hexdigest()
on_disk = {p.name: sha256(p) for p in CFG.MODEL_DIR.iterdir() if p.is_file()}
if on_disk != in_tar:
    bad = sorted(set(on_disk) ^ set(in_tar)) + sorted(k for k in set(on_disk) & set(in_tar)
                                                      if on_disk[k] != in_tar[k])
    stop(f"Extracted files in {CFG.MODEL_DIR} differ from the tarball: {bad[:10]}")
print(f"extracted model files for {CFG.ROUND}: {len(on_disk)} files, all identical to the tarball  OK")

datasets commit  68932db171f5e13349d9bda0dd9f96fcf6e227e2  == config.DATASETS_COMMIT  OK
LFS file         2024-07-21.ForecastBench.human_public_individual.json  sha256 matches PROVENANCE.md §2  OK


forecast_sets.tar.gz             sha256 219998293a577c820dfa4fdc34541833bd99fd47b60854c2e0424c1a38490b45  OK
processed_forecast_sets.tar.gz   sha256 7f20d24b16b5cacc0c9f1b4f09245aedb17f789504810402ba2458c6ca3486c7  OK


extracted model files for 2024-07-21: 141 files, all identical to the tarball  OK


## 2. Run directory

A fresh directory outside the repository holds everything the modules write: their reports, figures,
and `derived/` checkpoints. It is wiped at the start of every run, so nothing is reused.

In [3]:
RUN = CFG.STUDY_ROOT / "repro_v9"
MARK = RUN / ".repro_v9_rundir"
if RUN.exists():
    if not MARK.exists():
        raise RuntimeError(f"{RUN} exists and was not created by this notebook; move it away first.")
    shutil.rmtree(RUN)
DOCS_RUN, DER_RUN, LOGS = RUN / "docs", RUN / "derived", RUN / "logs"
for d in (DOCS_RUN / "figures", DER_RUN, LOGS):
    d.mkdir(parents=True)
MARK.write_text("created by notebooks/reproduce_v9.ipynb\n")
print("run directory", RUN)

# Point the in-kernel config at the run directory too (amendment4_h5 and the checks below use it).
CFG.DOCS, CFG.FIGDIR, CFG.DERIVED = DOCS_RUN, DOCS_RUN / "figures", DER_RUN

WRAPPER = """
import runpy, sys
from pathlib import Path
code, script, docs, derived = sys.argv[1:5]
sys.path.insert(0, code)
import config as CFG
CFG.DOCS = Path(docs); CFG.FIGDIR = CFG.DOCS / "figures"; CFG.DERIVED = Path(derived)
sys.argv = [script]
runpy.run_path(script, run_name="__main__")
"""
TIMES = {}


def run_script(name):
    """Run code/<name>.py unedited in a subprocess, outputs redirected to the run directory."""
    log = LOGS / f"{name}.log"
    t = time.time()
    with open(log, "w") as fh:
        r = subprocess.run([sys.executable, "-c", WRAPPER, str(CODE), str(CODE / f"{name}.py"),
                            str(DOCS_RUN), str(DER_RUN)],
                           cwd=REPO, env=ENV, stdout=fh, stderr=subprocess.STDOUT)
    TIMES[name] = time.time() - t
    if r.returncode != 0:
        tail = log.read_text().splitlines()[-15:]
        raise RuntimeError(f"{name}.py failed (exit {r.returncode}); log {log}:\n" + "\n".join(tail))
    print(f"{name}.py finished in {TIMES[name] / 60:.1f} min (log: {log})")


def report(name):
    return (DOCS_RUN / name).read_text()

run directory /Users/aaron/Desktop/forecast-study/repro_v9


## 3. Pipeline

### 3.1 SPEC v1: `full.py`
Takes about 20 minutes. Writes `forecast_full_REPORT.md` and `derived/full_frame.pkl`.

In [4]:
run_script("full")

full.py finished in 8.1 min (log: /Users/aaron/Desktop/forecast-study/repro_v9/logs/full.log)


### 3.2 SPEC v2: `v2_full.py`
Takes about 95 minutes. Writes `forecast_v2_full_REPORT.md` and the phase checkpoints
`derived/v2_full_*.pkl`. Amendment 6 reads its cross-fit checkpoint for its fold check.

In [5]:
run_script("v2_full")

v2_full.py finished in 94.4 min (log: /Users/aaron/Desktop/forecast-study/repro_v9/logs/v2_full.log)


### 3.3 Amendment 4: `amendment4_h5.py` (imported; `main()`)
This step adds the H5 question-level split section to the regenerated v1 report. The paper reports no
H5 number, so it is run for completeness. It asserts that the locked target-level split reproduces.

In [6]:
t = time.time()
import amendment4_h5 as A4
A4.main()
TIMES["amendment4_h5"] = time.time() - t
print(f"amendment4_h5.main() finished in {TIMES['amendment4_h5'] / 60:.1f} min")

       n_included  n_excluded  pearson_r     ci_lo     ci_hi  spearman_r  min_targets_in_smaller_half  target_level_r_locked
group                                                                                                                       
S              39           1   0.270156 -0.104021  0.524859    0.229352                           15               0.781768
P             500           0   0.215092  0.130985  0.295992    0.215715                            3               0.639996

         median      p2_5     p97_5       min       max  share_below_locked_target_r  max_excluded
group                                                                                             
S      0.375066 -0.447743  0.781327 -0.531518  0.848722                        0.975             1
P      0.229415  0.160087  0.292948  0.091521  0.341793                        1.000             1
amendment4_h5.main() finished in 0.1 min


### 3.4 SPEC v3 Amendments 5, 5b and 6

In [7]:
run_script("amendment5")
run_script("amendment5b")
run_script("amendment6")

amendment5.py finished in 0.5 min (log: /Users/aaron/Desktop/forecast-study/repro_v9/logs/amendment5.log)


amendment5b.py finished in 0.0 min (log: /Users/aaron/Desktop/forecast-study/repro_v9/logs/amendment5b.log)


amendment6.py finished in 0.4 min (log: /Users/aaron/Desktop/forecast-study/repro_v9/logs/amendment6.log)


## 4. Values from the regenerated reports

In [8]:
REC = {}          # key -> list of floats (or a str), as referenced by paper_numbers_v9.json
NUM = r"[+\-−]?\d[\d,]*(?:\.\d+)?"


def num(s):
    return float(str(s).replace("−", "-").replace(",", ""))


def nums(s):
    return [num(x) for x in re.findall(NUM, s)]


def grab(text, pattern):
    m = re.search(pattern, text, flags=re.S)
    if not m:
        raise KeyError(f"pattern not found in report: {pattern}")
    return m.groups()


def table_after(text, anchor, first_col, nth=0):
    """The nth markdown table after `anchor` whose first header cell is `first_col`."""
    lines = text[text.index(anchor):].splitlines()
    found = 0
    for i, line in enumerate(lines):
        if line.startswith(f"| {first_col} |") and i + 1 < len(lines) and lines[i + 1].startswith("|---"):
            if found < nth:
                found += 1
                continue
            hdr = [c.strip() for c in line.strip().strip("|").split("|")]
            rows = []
            for row in lines[i + 2:]:
                if not row.startswith("|"):
                    break
                cells = [c.strip() for c in row.strip().strip("|").split("|")]
                assert len(cells) == len(hdr), (anchor, row[:80])
                rows.append(cells)
            return pd.DataFrame(rows, columns=hdr)
    raise KeyError(f"no table with first column {first_col!r} after {anchor!r}")


# ---------------------------------------------------------------- v1 (full.py)
V1 = report("forecast_full_REPORT.md")
r = grab(V1, r"Total analysis frame: \*\*([\d,]+) forecaster × target rows\*\*, (\d+) resolved targets, "
             r"(\d+) questions")
REC["v1.n_rows"], REC["v1.n_targets"], REC["v1.n_questions"] = ([num(x)] for x in r)
r = grab(V1, r"Targets by family: \*\*(\d+) dataset, (\d+) market\*\* \((\d+) and (\d+) questions\)")
REC["v1.n_dataset"], REC["v1.n_market"], REC["v1.q_dataset"], REC["v1.q_market"] = ([num(x)] for x in r)
r = grab(V1, r"Forecasters per target — S: min (\d+), median (\d+), max \d+; P: min (\d+), median (\d+)")
REC["v1.fpt.S.min"], REC["v1.fpt.S.med"], REC["v1.fpt.P.min"], REC["v1.fpt.P.med"] = ([num(x)] for x in r)
cov = table_after(V1, "## 2. Data coverage", "index").set_index("index")
for g in ("S", "P"):
    REC[f"v1.cov.{g}.med"] = [num(cov.loc[g, "cov_med"])]
    REC[f"v1.cov.{g}.range"] = [num(cov.loc[g, "cov_min"]), num(cov.loc[g, "cov_max"])]
REC["v1.n_variants"] = [num(grab(V1, r"\*\*(\d+) variants\*\* = \d+ base models")[0])]
r = grab(V1, r"unmappable to a resolution date: \*\*([\d,]+)\*\*\. Human rows dropped as unresolved: "
             r"\*\*([\d,]+)\*\*")
REC["v1.unmappable"], REC["v1.unresolved"] = [num(r[0])], [num(r[1])]
r = grab(V1, r"\*\*(\d+) rows excluded\*\* from every analysis, from \*\*(\d+) distinct forecasters\*\*")
REC["v1.n_invalid"], REC["v1.invalid_forecasters"] = [num(r[0])], [num(r[1])]
REC["v1.invalid_max"] = [num(grab(V1, r"Values run to ([\d,]+\.\d+)\.")[0])]
r = grab(V1, r"\*\*([\d,]+) targets\*\* across (\d+) questions\. \*\*Intersection with the test set: (\d+)\*\*")
REC["v1.sel_targets"], REC["v1.sel_questions"], REC["v1.sel_intersection"] = ([num(x)] for x in r)
sel = table_after(V1, "## 3. Baseline (a)", "model")
assert sel.loc[0, "model"] == "Claude-3-5-Sonnet-20240620 (zero shot with freeze values)"
REC["v1.sel_brier_1"] = [100 * num(sel.loc[0, "brier"])]
REC["v1.sel_brier_2"] = [100 * num(sel.loc[1, "brier"])]
r = grab(V1, r"Eligibility \(Amendment 1 A\.3\): (\d+) of (\d+) ineligible")
REC["v1.n_ineligible"], REC["v1.n_eligible"] = [num(r[0])], [num(r[1]) - num(r[0])]
REC["v1.unrestricted_combo_pct"] = [num(grab(V1, r"targets, (\d+)% combination questions")[0])]
h1 = table_after(V1, "## 6. H1", "index").set_index("index")
h1c = table_after(V1, "### Sensitivity to Amendment 2 item 1", "index").set_index("index")
for g in ("S", "P"):
    REC[f"v1.h1.{g}"] = [num(h1.loc[g, c]) for c in ("mean", "ci_lo", "ci_hi")]
    REC[f"v1.h1clip.{g}"] = [num(h1c.loc[g, c]) for c in ("mean", "ci_lo", "ci_hi")]
h2_v1 = table_after(V1, "## 7. H2", "group").set_index("group")
h2s_v1 = table_after(V1, "## 7. H2", "group", nth=1).set_index("group")
for g in ("S", "P"):
    REC[f"v1.h2.{g}.clipped"] = [num(h2_v1.loc[g, "clipped_human"])]

# ---------------------------------------------------------------- Amendment 5
A5 = report("amendment5_REPORT.md")
REC["a5.Q_a"] = [100 * num(grab(A5, r"Benchmark \(a\) for comparison: Q = (0\.\d+)")[0])]
REC["a5.draws"] = [num(grab(A5, r"cluster bootstrap percentile interval \((\d+) draws")[0])]
q5 = table_after(A5, "Selection-set Q for every news variant", "model")
for _, row in q5.iterrows():
    REC[f"a5.Q.{row['model']}"] = [100 * num(row["Q_sel"])]
enc5 = table_after(A5, "## Encompassing fits", "model")
assert enc5["n_targets"].nunique() == 1
REC["a5.n_targets"] = [num(enc5["n_targets"].iloc[0])]
for _, row in enc5.iterrows():
    REC[f"a5enc.{row['model']}.{row['group']}.beta"] = nums(row["beta_news"])
    REC[f"a5enc.{row['model']}.{row['group']}.diff"] = nums(row["diff"])

# ---------------------------------------------------------------- Amendment 5b
A5B = report("amendment5b_REPORT.md")
p5b = table_after(A5B, "**Primary, clip (0.01, 0.99):**", "source")
for _, row in p5b.iterrows():
    REC[f"a5b.{row['source']}.{row['group']}"] = [num(row[c]) for c in ("coef_human_minus_model", "ci_lo", "ci_hi")]
    REC[f"a5b.{row['source']}.n"] = [num(row["n_targets"])]

# ---------------------------------------------------------------- Amendment 6
A6 = report("amendment6_REPORT.md")
r = grab(A6, r"(\d+) targets on (\d+) questions; one row per target")
REC["a6.n_targets"], REC["a6.n_questions"] = [num(r[0])], [num(r[1])]
REC["a6.mens_n"] = [num(grab(A6, r"Mens = median of the (\d+) matched variants other than M1")[0])]
REC["a6.n_folds"] = [num(grab(A6, r"Folds: v2 cross-fit folds \((\d+), by question")[0])]
REC["a6.draws"] = [num(grab(A6, r"Intervals: question-level cluster bootstrap, (\d+) draws")[0])]
REC["a6.noise_k"] = [num(grab(A6, r"Noise control N7: median of (\d+) public forecasts")[0])]
REC["a6.noise_draws"] = [num(grab(A6, r"; (\d+) draws give the median")[0])]
REC["a6.sd_rule"] = [num(grab(A6, r"fold-to-fold SD exceeds (0\.\d+)")[0])]

sys6 = table_after(A6, "## Systems: pooled out-of-sample scores", "system")
for _, row in sys6.iterrows():
    lg, br = num(row["log_score"]), num(row["brier"])
    REC[f"a6sys.{row['system']}.log"], REC[f"a6sys.{row['system']}.both"] = [lg], [lg, br]

CMP6 = {}
for m in re.finditer(r"^\| comparison \| d_log \| log_ci \| d_brier \| brier_ci \|$", A6, flags=re.M):
    tab = table_after(A6[m.start():], "| comparison |", "comparison")
    for _, row in tab.iterrows():
        k = re.sub(r"\s+vs\s+", " vs ", row["comparison"]).strip()
        CMP6[k] = row
        REC[f"a6cmp.{k}.log"] = [num(row["d_log"]), *nums(row["log_ci"])]
        REC[f"a6cmp.{k}.brier"] = [num(row["d_brier"]), *nums(row["brier_ci"])]

for c in ("C1", "C2"):
    r = grab(A6, rf"H_S alone vs H_S\+M1 · {c}: log ({NUM}) \[({NUM}), ({NUM})\], "
                 rf"Brier ({NUM}) \[({NUM}), ({NUM})\]")
    REC[f"a6note.H_S vs H_S+M1 · {c}.log"] = [num(x) for x in r[:3]]
    REC[f"a6note.H_S vs H_S+M1 · {c}.brier"] = [num(x) for x in r[3:]]

wt6 = table_after(A6, "## Fitted weights", "system")
for _, row in wt6.iterrows():
    j = {"w(first)": "1", "w(second)": "2"}[row["weight"]]
    REC[f"a6w.{row['system']}.mean{j}"] = [num(row["mean"])]
    if j == "1":
        REC[f"a6w.{row['system']}.min"] = [num(row["min"])]

for g in ("H_S", "H_P"):
    d = [REC[f"a6cmp.{g}+Mens · {c} vs Mens.log"][0] for c in ("C1", "C2", "C3", "C4")]
    REC[f"a6coprim.{g}.range"] = [min(d), max(d)]
band = table_after(A6, "Across 200 draws", "combiner")
band = band[(band["comparison"] == "N7 + Mens vs Mens") & band["combiner"].isin(["C1", "C2", "C3", "C4"])]
assert len(band) == 4
lims = np.array([nums(s)[1:] for s in band["n7"]])
REC["a6noise.envelope"] = [lims[:, 0].min(), lims[:, 1].max()]
print(f"{len(REC)} values read from the regenerated reports")

392 values read from the regenerated reports


## 5. Checks the modules do not print

### 5.1 Encompassing in both directions (Table 2, §5.2, Table A4)
The v1 H2 fit `o ~ logit(p_a) + b_h·[logit(p_h) − logit(p_a)]` equals
`o ~ (b_a − b_h)·logit(p_a) + b_h·logit(p_h)`, so the coefficient on M1 given the human median is
`b_a − b_h`. Its interval comes from the same fit's question-clustered covariance. The fit repeats
`models.encompassing` only to keep the covariance. Its coefficients and intervals are asserted equal to
`models.encompassing`, and those are asserted equal to the regenerated v1 and Amendment 5b reports.

In [9]:
import fbdata as F
import models as M
import v2common as V

full = pd.read_pickle(DER_RUN / "full_frame.pkl")
tl = V.target_level(full)
mkt = full.groupby("target")["MKT"].first()
tl["subset"] = np.where(tl["target"].map(mkt) == 1, "market", "dataset")


def enc_both(t, col, clip):
    ref = M.encompassing(t, col, clip)
    d = t.dropna(subset=[col, "p_a", "o"])
    la, _ = M.logit_clip(d["p_a"].values, *clip)
    lh, _ = M.logit_clip(d[col].values, *clip)
    X = sm.add_constant(pd.DataFrame({"logit_p_a": la, "human_minus_model": lh - la}, index=d.index))
    m = sm.Logit(d["o"].values, X).fit(disp=0, cov_type="cluster",
                                       cov_kwds={"groups": d["question"].values, "use_correction": True})
    assert np.allclose(np.asarray(m.params), ref["params"].values, rtol=0, atol=1e-10)
    assert np.allclose(np.asarray(m.conf_int()), ref["ci"].values, rtol=0, atol=1e-10)
    tt = m.t_test(np.array([[0.0, 1.0, -1.0]]))
    lo, hi = np.asarray(tt.conf_int(alpha=0.05))[0]
    bh = ref["params"]["human_minus_model"]
    return dict(bh=[bh, ref["ci"].loc["human_minus_model", 0], ref["ci"].loc["human_minus_model", 1]],
                rev=[float(np.squeeze(tt.effect)), float(lo), float(hi)],
                n=[ref["nobs"]], ncl=[ref["n_clusters"]], pa=ref["params"]["logit_p_a"])


ENC = {}
for clip, tag in ((CFG.CLIP, "h2"), ((0.001, 0.999), "h2s")):
    for g in ("S", "P"):
        e = enc_both(tl, f"p_h_{g}", clip)
        ENC[(tag, g)] = e
        for k in ("bh", "rev", "n", "ncl"):
            REC[f"{tag}.{g}.{k}"] = e[k]
        assert abs(e["rev"][0] - (e["pa"] - e["bh"][0])) < 1e-12
        v1tab = h2_v1 if tag == "h2" else h2s_v1
        assert [f"{x:.4f}" for x in e["bh"]] == [v1tab.loc[g, c] for c in
                                                 ("coef_human_minus_model", "ci_lo", "ci_hi")], (tag, g)
for sub in ("dataset", "market"):
    for g in ("S", "P"):
        e = enc_both(tl[tl["subset"] == sub], f"p_h_{g}", CFG.CLIP)
        for k in ("rev", "n", "ncl"):
            REC[f"rev.{sub}.{g}.{k}"] = e[k]
        row5b = p5b[(p5b["source"] == sub) & (p5b["group"] == g)].iloc[0]
        assert [f"{x:.2f}" for x in e["bh"]] == [row5b[c] for c in
                                                 ("coef_human_minus_model", "ci_lo", "ci_hi")], (sub, g)

print(pd.DataFrame({f"{tag} {g}": {"b_h": "%.4f [%.4f, %.4f]" % tuple(e["bh"]),
                                   "b_a - b_h": "%.4f [%.4f, %.4f]" % tuple(e["rev"])}
                    for (tag, g), e in ENC.items()}).T.to_string())

                           b_h                  b_a - b_h
h2 S   1.7892 [1.0444, 2.5341]  -0.2309 [-0.7654, 0.3035]
h2 P   0.9718 [0.2397, 1.7040]    0.7242 [0.3247, 1.1237]
h2s S  1.7650 [0.9908, 2.5391]  -0.2399 [-0.7723, 0.2925]
h2s P  0.9715 [0.2391, 1.7039]    0.7240 [0.3247, 1.1233]


### 5.2 Amendment 6 point fit, in the kernel
`amendment6.py` is a script. Its top-level statements up to the bootstrap (the first assignment to
`idx_q`) are executed unedited in a namespace. That gives the unrounded point-fit objects: the sources,
folds, combiners and fold weights. The rounded scores are asserted equal to the regenerated report.

In [10]:
a6_path = CODE / "amendment6.py"
tree = ast.parse(a6_path.read_text())
cut = next(i for i, n in enumerate(tree.body) if isinstance(n, ast.Assign)
           and any(getattr(t, "id", None) == "idx_q" for t in n.targets))
A6NS = {"__name__": "amendment6_point_fit", "__file__": str(a6_path)}
exec(compile(ast.Module(body=tree.body[:cut], type_ignores=[]), str(a6_path), "exec"), A6NS)

NAMES = {"Hs": "H_S", "Hp": "H_P"}
def a6_label(s):
    name, _, c = s.partition("|")
    name = "+".join(NAMES.get(x, x) for x in name.split("+"))
    return f"{name} · {c}" if c else name

sys6i = sys6.set_index("system")
for s in A6NS["SYSTEMS"]:
    assert [f"{A6NS['LOG'][s]:.3f}", f"{A6NS['BRI'][s]:.3f}"] == list(sys6i.loc[a6_label(s), ["log_score", "brier"]]), s

WT = A6NS["WT"]
con = WT[WT.system.str.endswith("|C3") | WT.system.str.endswith("|C4")].sort_values("sd", ascending=False)
REC["nb.a6_max_sd"] = [float(con["sd"].iloc[0])]
REC["nb.a6_max_sd_system"] = a6_label(con["system"].iloc[0])
print("largest constrained fold SDs (unrounded):")
print(con.head(3).assign(system=con["system"].map(a6_label))[["system", "weight", "mean", "sd"]]
      .to_string(index=False, float_format="%.6f"))

# Matched-condition coverage of the test targets (§2.2)
mods, variants, tl6 = A6NS["mods"], A6NS["variants"], A6NS["tl"]
test = set(tl6["target"])
have = mods[mods["model"].isin(variants) & mods["target"].isin(test)].groupby("model")["target"].nunique()
REC["nb.cov_missing"] = [float(sum(len(test) - have.get(m, 0) for m in variants))]
meta = mods[mods["model"].isin(variants)].drop_duplicates("model")
REC["nb.n_base"] = [float(meta["base"].nunique())]
REC["nb.n_scaffolds"] = [float(meta["scaffold"].nunique())]
REC["nb.invalid_share_pct"] = [100 * REC["v1.n_invalid"][0] / (len(full) + REC["v1.n_invalid"][0])]
print(f"\n{len(variants)} matched variants, {meta['base'].nunique()} base models, "
      f"{meta['scaffold'].nunique()} scaffolds; variant-target pairs missing on the test set: "
      f"{int(REC['nb.cov_missing'][0])}")

largest constrained fold SDs (unrounded):
       system   weight     mean       sd
  H_P+M1 · C3 w(first) 0.475766 0.099323
H_P+Mens · C3 w(first) 0.788628 0.099128
H_P+Mens · C4 w(first) 0.779946 0.094753

34 matched variants, 17 base models, 2 scaffolds; variant-target pairs missing on the test set: 0


### 5.3 Superforecaster ids (§2.1)
The counts come from the individual human files as published (before the resolution filter) and from
the analysis frame. The ids themselves are not printed.

In [11]:
raw = {g: pd.DataFrame(json.loads(p.read_text())["forecasts"]) for g, p in CFG.HUMAN_FILES.items()}
for g, d in raw.items():
    d["question"] = d["source"] + "|" + d["id"].astype(str)
REC["ids.S_n"] = [float(raw["S"]["user_id"].nunique())]
REC["ids.P_n"] = [float(raw["P"]["user_id"].nunique())]
REC["ids.questions"] = [float(pd.concat([raw["S"]["question"], raw["P"]["question"]]).nunique())]
per_id = raw["S"].groupby("user_id").agg(n=("question", "size"), nq=("question", "nunique"),
                                         n_mkt=("source", lambda s: sum(F._is_market(x) for x in s)))
per_id = per_id.sort_values("n")
REC["ids.S_min_forecasts"] = [float(per_id["n"].iloc[0])]
REC["ids.S_min_market"] = [float(per_id["n_mkt"].iloc[0])]
REC["ids.S_second_min"] = [float(per_id["n"].iloc[1])]
REC["ids.S_max_questions"] = [float(per_id["nq"].max())]
fs = full[full.GRP == "S"]
tg_per_id = fs.groupby("forecaster")["target"].nunique()
REC["ids.S_n_frame"] = [float(fs["forecaster"].nunique())]
REC["ids.S_max_targets_frame"] = [float(tg_per_id.max())]
smallest_id, all_targets_id = per_id.index[0], tg_per_id.idxmax()
assert (tg_per_id == tg_per_id.max()).sum() == 1


def median_without(fid):
    """Superforecaster median per target from the analysis frame without one id, aligned to `tl`."""
    m = fs[fs["forecaster"] != fid].groupby("target")["p_h"].median()
    return tl["target"].map(m).to_numpy(float)


def n_changed(new):
    return float(np.sum(~np.isclose(new, tl["p_h_S"].to_numpy(float), rtol=0, atol=1e-12)))


REC["ids.min_id_median_changes"] = [n_changed(median_without(smallest_id))]
print(f"superforecaster ids {int(REC['ids.S_n'][0])}, public ids {int(REC['ids.P_n'][0])}, "
      f"questions {int(REC['ids.questions'][0])}")
print(f"forecasts per superforecaster id: smallest {int(per_id['n'].iloc[0])} "
      f"({int(per_id['n_mkt'].iloc[0])} on market questions), next {int(per_id['n'].iloc[1])}; "
      f"largest question count {int(per_id['nq'].max())}")
print(f"the smallest id changes the median on {int(REC['ids.min_id_median_changes'][0])} of {len(tl)} targets")

superforecaster ids 40, public ids 500, questions 189
forecasts per superforecaster id: smallest 4 (4 on market questions), next 63; largest question count 189
the smallest id changes the median on 2 of 578 targets


### 5.4 Leave-one-id-out (footnote 1)
The superforecaster median is rebuilt without the one id present on every target. M1, the folds, the
clip and the combiners are unchanged: the v1 H2 fit through `models.encompassing`, and the Amendment 6
log score and fitted C3/C4 weights through the Amendment 6 functions.

In [12]:
hs_loo = median_without(all_targets_id)
REC["loo.a.changed"] = [n_changed(hs_loo)]
tl_loo = tl.assign(p_h_S=hs_loo)
e = M.encompassing(tl_loo, "p_h_S", CFG.CLIP)
REC["loo.a.bh"] = [e["params"]["human_minus_model"], e["ci"].loc["human_minus_model", 0],
                   e["ci"].loc["human_minus_model", 1]]

# align to the Amendment 6 target order
assert (tl6["target"].to_numpy() == tl["target"].to_numpy()).all()
assert np.allclose(A6NS["src"]["Hs"], tl["p_h_S"].to_numpy(float), rtol=0, atol=0)
src_loo = dict(A6NS["src"], Hs=hs_loo)
y6, folds6 = A6NS["y"], A6NS["folds"]
pred, wts = A6NS["oos"](src_loo, y6, folds6, ["Hs", "M1", "Hs+M1|C3", "Hs+M1|C4"])
lg, _ = A6NS["scores"](pred, y6)
REC["loo.a.dlog"] = [lg["Hs"] - lg["M1"]]
for c in ("C3", "C4"):
    REC[f"loo.a.w.{c}.min"] = [float(np.min(wts[f"Hs+M1|{c}"]))]
print(f"without the all-targets id: median changes on {int(REC['loo.a.changed'][0])} targets; "
      "H2 b_h %.4f [%.4f, %.4f]; " % tuple(REC["loo.a.bh"])
      + f"H_S vs M1 log {REC['loo.a.dlog'][0]:+.4f}; fold weights C3 "
      + ", ".join(f"{x:.6f}" for x in wts["Hs+M1|C3"]) + "; C4 "
      + ", ".join(f"{x:.6f}" for x in wts["Hs+M1|C4"]))

without the all-targets id: median changes on 316 targets; H2 b_h 1.6803 [0.9410, 2.4196]; H_S vs M1 log +0.0963; fold weights C3 1.000000, 1.000000, 1.000000, 1.000000, 1.000000; C4 0.999999, 0.999999, 0.999999, 1.000000, 0.999999


### 5.5 Provenance and configuration values the paper states

In [13]:
REC["prov.commit7"] = head[:7]
REC["cfg.round"] = CFG.ROUND
REC["cfg.clip"] = list(CFG.CLIP)
REC["cfg.imputed_max_pct"] = [100 * CFG.IMPUTED_ELIGIBILITY_MAX]
print(f"{len(REC)} recomputed values in total")

445 recomputed values in total


## 6. Number-by-number check

In [14]:
SPEC = json.loads((REPO / "notebooks" / "paper_numbers_v9.json").read_text())


def decimals(tok):
    tok = str(tok)
    return len(tok.split(".")[1]) if "." in tok else 0


def fmt(x, d):
    return f"{x:.{d}f}".replace("-", "−")


rows = []
for e in SPEC["numbers"]:
    rec = REC[e["key"]]
    if e.get("type") == "str":
        want = e.get("values", [e["paper"]])[0]
        rows.append(dict(id=e["id"], section=e["section"], label=e["label"], paper=e["paper"],
                         recomputed=rec, raw=rec, match=rec == want))
        continue
    if "values" in e:
        pv, pd_ = e["values"], [decimals(v) for v in e["values"]]
    else:
        toks = re.findall(NUM, e["paper"])
        pv, pd_ = [num(t) for t in toks], [decimals(t) for t in toks]
    rv = list(rec)
    if e.get("transform") == "neg":
        rv = [-rv[0], -rv[2], -rv[1]] if len(rv) == 3 else [-x for x in rv]
    if e.get("scale"):
        rv = [x / e["scale"] for x in rv]
    if len(pv) == 1 and len(rv) > 1:
        rv = rv[:1]
    assert len(pv) == len(rv), (e["id"], pv, rv)
    tol = [e["tol"]] * len(pv) if "tol" in e else [0.5 * 10 ** -d for d in pd_]
    ok = all(abs(r - p) <= t + 1e-9 for r, p, t in zip(rv, pv, tol))
    rows.append(dict(id=e["id"], section=e["section"], label=e["label"], paper=e["paper"],
                     recomputed=", ".join(fmt(r, d) for r, d in zip(rv, pd_)),
                     raw=", ".join(f"{r:.6g}" for r in rv), match=bool(ok)))

CHECK = pd.DataFrame(rows).set_index("id")
n_ok = int(CHECK["match"].sum())
print(f"paper numbers checked: {len(CHECK)}; match: {n_ok}; do not match: {len(CHECK) - n_ok}")
print(f"numbers in the paper not recomputed (cited or procedural): {len(SPEC['not_recomputed'])}")
print(f"total runtime {(time.time() - T_START) / 60:.0f} min; "
      + ", ".join(f"{k} {v / 60:.1f} min" for k, v in TIMES.items()))

paper numbers checked: 297; match: 295; do not match: 2
numbers in the paper not recomputed (cited or procedural): 11
total runtime 104 min; full 8.1 min, v2_full 94.4 min, amendment4_h5 0.1 min, amendment5 0.5 min, amendment5b 0.0 min, amendment6 0.4 min


### Not recomputed
These numbers in the paper are cited results, ForecastBench procedure, dates or bibliographic details.
They have no counterpart in the data.

In [15]:
print(pd.DataFrame(SPEC["not_recomputed"]).to_string(index=False))

                               section                                                                 text                                                                    reason
                             §1 (p. 1)                          twelve language models; 31 binary questions                                   cited result (Schoenegger et al., 2024)
                             §1 (p. 1)                                         2024 rounds of ForecastBench                                        cited result (Karger et al., 2025)
                           §2.1 (p. 2)                                         snapshot fixed on 2026-09-06 provenance date; the hashes and commit are checked in the provenance cell
                           §2.1 (p. 2)            39 superforecasters recruited (Karger et al., 2025, p. 7)                                                               cited count
                           §2.1 (p. 2)   question sets ten days before the due date; model

### Match table

In [16]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70, "display.width", 250):
    print(CHECK[["section", "label", "paper", "recomputed", "match"]].to_string())

                         section                                                                                                              label                           paper              recomputed  match
id                                                                                                                                                                                                                
N001             Abstract (p. 1)                                                                                                superforecaster ids                              40                      40   True
N002             Abstract (p. 1)                                                                                                 public forecasters                             500                     500   True
N003             Abstract (p. 1)                                                                             H2 coefficient, superforecaster median         

### Rows that do not match

In [17]:
BAD = CHECK[~CHECK["match"]]
if BAD.empty:
    print(f"All {len(CHECK)} rows match.")
else:
    print(f"{len(BAD)} of {len(CHECK)} rows do NOT match:\n")
    with pd.option_context("display.max_colwidth", 120, "display.width", 250):
        print(BAD[["section", "label", "paper", "recomputed", "raw"]].to_string())

2 of 297 rows do NOT match:

          section                                                                                  label                paper          recomputed                             raw
id                                                                                                                                                                               
N093  §5.2 (p. 5)                                              clip [0.001, 0.999]: superforecasters b_h    1.77 [0.99, 2.54]    1.76, 0.99, 2.54      1.76499, 0.990833, 2.53914
N095  §5.2 (p. 5)  clip [0.001, 0.999]: superforecasters reverse ('unchanged to two decimals' = Table 2)  −0.23 [−0.77, 0.30]  −0.24, −0.77, 0.29  -0.239926, -0.772314, 0.292461
